In [ ]:
!mkdir -p eval/out

!yosys -q -p "read_verilog eval/handcrafted/bad_multiplier.v; proc; opt_merge; opt_clean; write_json eval/out/bad_multiplier.json; synth_xilinx -family xcup; tee -o eval/out/bad_multiplier.stat stat"
!yosys -q -p "read_verilog eval/handcrafted/complex_multiplier.v; proc; opt_merge; opt_clean; write_json eval/out/complex_multiplier.json; synth_xilinx -family xcup; tee -o eval/out/complex_multiplier.stat stat"
!yosys -q -p "read_verilog eval/handcrafted/dot_product.v; proc; opt_merge; opt_clean; write_json eval/out/dot_product.json; synth_xilinx -family xcup; tee -o eval/out/dot_product.stat stat"
!yosys -q -p "read_verilog eval/handcrafted/multiplier_with_rst.v; proc; opt_merge; opt_clean; write_json eval/out/multiplier_with_rst.json; synth_xilinx -family xcup; tee -o eval/out/multiplier_with_rst.stat stat"
!yosys -q -p "read_verilog eval/handcrafted/redundant_adders.v; proc; opt_merge; opt_clean; write_json eval/out/redundant_adders.json; synth_xilinx -family xcup; tee -o eval/out/redundant_adders.stat stat"
!yosys -q -p "read_verilog eval/handcrafted/signed_mac.v; proc; opt_merge; opt_clean; write_json eval/out/signed_mac.json; synth_xilinx -family xcup; tee -o eval/out/signed_mac.stat stat"
!yosys -q -p "read_verilog eval/handcrafted/signed_reg.v; proc; opt_merge; opt_clean; write_json eval/out/signed_reg.json; synth_xilinx -family xcup; tee -o eval/out/signed_reg.stat stat"
!yosys -q -p "read_verilog eval/handcrafted/square_diff.v; proc; opt_merge; opt_clean; write_json eval/out/square_diff.json; synth_xilinx -family xcup; tee -o eval/out/square_diff.stat stat"
!yosys -q -p "read_verilog eval/handcrafted/unsigned_mac.v; proc; opt_merge; opt_clean; write_json eval/out/unsigned_mac.json; synth_xilinx -family xcup; tee -o eval/out/unsigned_mac.stat stat"
!yosys -q -p "read_verilog eval/handcrafted/wide_multiplier.v; proc; opt_merge; opt_clean; write_json eval/out/wide_multiplier.json; synth_xilinx -family xcup; tee -o eval/out/wide_multiplier.stat stat"

In [ ]:
import emap
import json

SCHEMA_PATH = "emap/schema.sql"

def simple_cost_model(type_: str, *ports) -> float:
    if type_ == "$dff":
        return len(ports[0]) * 1.0
    elif type_ in {"$muls", "$mulu"}:
        return len(ports[0]) * len(ports[1]) * 1.0
    elif type_ in {"$adds", "$addu", "$subs", "$subu"}:
        return min(len(ports[0]) + len(ports[1]), len(ports[2])) * 1.0
    return len(ports[0]) * 1.0  # other types

dsp_rules = {
    "signed_mul_1_stage_26_17_48_bit": {    # rule name
        "requirements": {                   # resource requirements
            "dsp48e2": 1                    # use one DSP48E2
        },
        "hidden_inputs": ["clk"],   # hidden input ports, e.g., clock
        "inputs": ["a", "b"],       # input ports
        "outputs": ["p"],           # output ports
        # and a match pattern in SQL
        "match_sql": """
            SELECT mul1.a, mul1.b, dff1.q
            FROM dffs AS dff1 JOIN aby_cells AS mul1
            ON dff1.d = mul1.y
            WHERE mul1.type = '$muls'
                AND width_of(mul1.a) <= 26 AND width_of(mul1.b) <= 17 AND width_of(dff1.q) <= 48
        """
    },
    "signed_muladd_1_stage_26_17_48_bit": {
        "requirements": {
            "dsp48e2": 1
        },
        "hidden_inputs": ["clk"],
        "inputs": ["a", "b", "c"],
        "outputs": ["p"],
        "match_sql": """
            SELECT mul1.a, mul1.b, add1.b, dff1.q
            FROM dffs AS dff1 JOIN aby_cells AS mul1 JOIN aby_cells AS add1
            ON dff1.d = add1.y AND mul1.y = add1.a
            WHERE mul1.type = '$muls' AND add1.type = '$adds'
                AND width_of(mul1.a) <= 26 AND width_of(mul1.b) <= 17 AND width_of(add1.b) <= 48 AND width_of(dff1.q) <= 48
        """
    },
    "unsigned_muladd_1_stage_27_18_48_bit": {
        "requirements": {
            "dsp48e2": 1
        },
        "hidden_inputs": ["clk"],
        "inputs": ["a", "b", "c"],
        "outputs": ["p"],
        "match_sql": """
            SELECT mul1.a, mul1.b, add1.b, dff1.q
            FROM dffs AS dff1 JOIN aby_cells AS mul1 JOIN aby_cells AS add1
            ON dff1.d = add1.y AND mul1.y = add1.a
            WHERE mul1.type = '$mulu' AND add1.type = '$addu'
                AND width_of(mul1.a) <= 27 AND width_of(mul1.b) <= 18 AND width_of(add1.b) <= 48 AND width_of(dff1.q) <= 48
        """
    },
    "signed_mulsub_1_stage_27_18_48_bit": {
        "requirements": {
            "dsp48e2": 1
        },
        "hidden_inputs": ["clk"],
        "inputs": ["a", "b", "c"],
        "outputs": ["p"],
        "match_sql": """
            SELECT mul1.a, mul1.b, sub1.b, dff1.q
            FROM dffs AS dff1 JOIN aby_cells AS mul1 JOIN aby_cells AS sub1
            ON dff1.d = sub1.y AND mul1.y = sub1.a
            WHERE mul1.type = '$muls' AND sub1.type = '$subs'
                AND width_of(mul1.a) <= 27 AND width_of(mul1.b) <= 18 AND width_of(sub1.b) <= 48 AND width_of(dff1.q) <= 48
        """
    },
    "signed_submuladd_1_stage_26_18_48_26_bit": {
        "requirements": {
            "dsp48e2": 1
        },
        "hidden_inputs": ["clk"],
        "inputs": ["d", "a", "b", "c"],
        "outputs": ["p"],
        "match_sql": """
            SELECT sub1.a, sub1.b, mul1.b, add1.b, dff1.q
            FROM dffs AS dff1 JOIN aby_cells AS sub1 JOIN aby_cells AS mul1 JOIN aby_cells AS add1
            ON dff1.d = add1.y AND sub1.y = mul1.a AND mul1.y = add1.a
            WHERE sub1.type = '$subs' AND mul1.type = '$muls' AND add1.type = '$adds'
                AND width_of(sub1.a) <= 26 AND width_of(sub1.b) <= 26 AND width_of(mul1.b) <= 18 AND width_of(add1.b) <= 48 AND width_of(dff1.q) <= 48
        """
    },
    "signed_addmuladd_1_stage_26_18_48_26_bit": {
        "requirements": {
            "dsp48e2": 1
        },
        "hidden_inputs": ["clk"],
        "inputs": ["a", "d", "b", "c"],
        "outputs": ["p"],
        "match_sql": """
            SELECT add2.a, add2.b, mul1.b, add1.b, dff1.q
            FROM dffs AS dff1 JOIN aby_cells AS add2 JOIN aby_cells AS mul1 JOIN aby_cells AS add1
            ON dff1.d = add1.y AND add2.y = mul1.a AND mul1.y = add1.a
            WHERE add2.type = '$adds' AND mul1.type = '$muls' AND add1.type = '$adds'
                AND width_of(add2.a) <= 26 AND width_of(add2.b) <= 26 AND width_of(mul1.b) <= 18 AND width_of(add1.b) <= 48 AND width_of(dff1.q) <= 48
        """
    },
    "signed_submul_1_stage_27_18_48_bit": {
        "requirements": {
            "dsp48e2": 1
        },
        "hidden_inputs": ["clk"],
        "inputs": ["d", "a", "b"],
        "outputs": ["p"],
        "match_sql": """
            SELECT sub1.a, sub1.b, mul1.b, dff1.q
            FROM dffs AS dff1 JOIN aby_cells AS sub1 JOIN aby_cells AS mul1
            ON dff1.d = mul1.y AND sub1.y = mul1.a
            WHERE sub1.type = '$subs' AND mul1.type = '$muls'
                AND width_of(sub1.a) <= 27 AND width_of(sub1.b) <= 27 AND width_of(mul1.b) <= 18 AND width_of(dff1.q) <= 48
        """
    },
    "signed_mul_2_stage_26_17_48_bit_rst": {
        "requirements": {
            "dsp48e2": 1
        },
        "hidden_inputs": ["clk"],
        "inputs": ["a", "b", "rst"],
        "outputs": ["p"],
        "match_sql": """
            SELECT sdff_a.d, sdff_b.d, sdff_p.rst, sdff_p.q
            FROM sdffs AS sdff_a JOIN sdffs AS sdff_b JOIN aby_cells AS mul JOIN sdffs AS sdff_p
            ON sdff_a.q = mul.a AND sdff_b.q = mul.b AND mul.y = sdff_p.d
            WHERE mul.type = '$muls'
                AND width_of(sdff_a.d) <= 26 AND width_of(sdff_b.d) <= 17 AND width_of(sdff_p.q) <= 48
                AND sdff_a.rst = sdff_b.rst AND sdff_b.rst = sdff_p.rst
                AND sdff_a.rst_val = 0 AND sdff_b.rst_val = 0 AND sdff_p.rst_val = 0
        """
    },
    "signed_square_diff_1_stage_18_bit": {
        "requirements": {
            "dsp48e2": 1
        },
        "hidden_inputs": ["clk"],
        "inputs": ["a", "d"],
        "outputs": ["p"],
        "match_sql": """
            SELECT sub1.a, sub1.b, dff1.q
            FROM dffs AS dff1 JOIN aby_cells AS sub1 JOIN aby_cells AS mul1
            ON dff1.d = mul1.y AND sub1.y = mul1.a AND sub1.y = mul1.b
            WHERE sub1.type = '$subs' AND mul1.type = '$muls'
                AND width_of(sub1.a) <= 18 AND width_of(sub1.b) <= 18 AND width_of(mul1.a) <= 18 AND width_of(dff1.q) <= 36
        """
    }
}

In [ ]:
TEST_NAME = "bad_multiplier"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()
cnt = 1
while cnt > 0:
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$mulu"])

    cnt = emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

mod = emap.extracts.ilp.extract_no_techmap(netlist, simple_cost_model, solver_type='auto', OutputFlag=False)

with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

In [ ]:
TEST_NAME = "complex_multiplier"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()

complex_mul_matches = emap.rewrites.ematch_complex_mul(netlist)
cnt = emap.rewrites.apply_complex_mul(netlist, complex_mul_matches)
print(f"Applied {cnt} rewrites")
netlist.rebuild()
cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds", "$muls"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$muls", "$subs"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls", "$subs"])

    cnt = emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)
    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.rewrite_tech(netlist, dsp_rules)
mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 3}, solver_type='auto', OutputFlag=False)

with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

In [ ]:
TEST_NAME = "dot_product"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()

cnt = 1
while cnt > 0:
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$muls"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls"])

    cnt = emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)
    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.rewrite_tech(netlist, dsp_rules)
mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 3}, solver_type='auto', OutputFlag=False)

with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

In [ ]:
TEST_NAME = "multiplier_with_rst"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()

cnt = emap.rewrites.rewrite_sdff(netlist)   # rewrite $dff to $sdff
print(f"Applied {cnt} rewrites")

# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.rewrite_tech(netlist, dsp_rules)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 1}, solver_type='auto', OutputFlag=False)
with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

In [ ]:
TEST_NAME = "redundant_adders"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()

cnt = 1
while cnt > 0:
    unsigned_add_matches = emap.rewrites.select_aby_cell_by_type(netlist, ["$addu"])
    cnt = emap.rewrites.apply_unsigned_add_bitblast(netlist, ((a, b, y) for _, a, b, y in unsigned_add_matches))
    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

mod = emap.extracts.ilp.extract_no_techmap(netlist, simple_cost_model, solver_type='auto', OutputFlag=False)
with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

In [ ]:
TEST_NAME = "signed_mac"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()

cnt = 1
while cnt > 0:
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$muls"])
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds", "$muls"])

    cnt = emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.rewrite_tech(netlist, dsp_rules)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 1}, solver_type='auto', OutputFlag=False)
with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

In [ ]:
TEST_NAME = "signed_reg"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()
dff_word_matches = emap.rewrites.ematch_word_dff(netlist)
cnt = emap.rewrites.apply_word_dff_split(netlist, dff_word_matches)
print(f"Applied {cnt} rewrites")
netlist.rebuild()

mod = emap.extracts.ilp.extract_no_techmap(netlist, simple_cost_model, solver_type='auto', OutputFlag=False)

with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

In [ ]:
TEST_NAME = "square_diff"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()

cnt = 1
while cnt > 0:
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$subs", "$muls"])

    cnt = emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)
    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.rewrite_tech(netlist, dsp_rules)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 1}, solver_type='auto', OutputFlag=False)
with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

In [ ]:
TEST_NAME = "unsigned_mac"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()

cnt = 1
while cnt > 0:
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$addu", "$mulu"])
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$addu", "$mulu"])

    cnt = emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.rewrite_tech(netlist, dsp_rules)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 1}, solver_type='auto', OutputFlag=False)
with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

In [ ]:
TEST_NAME = "wide_multiplier"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"eval/out/{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()
wide_mulu_matches = emap.rewrites.ematch_wide_mulu(netlist, a_width=17, b_width=26)
cnt = emap.rewrites.apply_wide_mulu_split(netlist, wide_mulu_matches, a_width=17, b_width=26)
print(f"Applied {cnt} rewrites")

cnt = 1
while cnt > 0:
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$addu", "$mulu"])
    wide_dff_matches = emap.rewrites.ematch_wide_dff(netlist, width_threshold=17)
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$addu", "$mulu"])

    cnt = emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)
    cnt += emap.rewrites.apply_wide_dff_split(netlist, wide_dff_matches, width_threshold=17)
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.rewrite_tech(netlist, dsp_rules)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 1}, solver_type='auto', OutputFlag=False)
with open(f"eval/out/{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

In [ ]:
!yosys -q -p "read_json eval/out/bad_multiplier_extracted.json; synth_xilinx -family xcup; tee -o eval/out/bad_multiplier_extracted.stat stat"
!yosys -q -p "read_json eval/out/complex_multiplier_extracted.json; read_verilog eval/blackboxes/dsp_defs.v; synth_xilinx -family xcup; tee -o eval/out/complex_multiplier_extracted.stat stat"
!yosys -q -p "read_json eval/out/dot_product_extracted.json; read_verilog eval/blackboxes/dsp_defs.v; synth_xilinx -family xcup; tee -o eval/out/dot_product_extracted.stat stat"
!yosys -q -p "read_json eval/out/multiplier_with_rst_extracted.json; read_verilog eval/blackboxes/dsp_defs.v; synth_xilinx -family xcup; tee -o eval/out/multiplier_with_rst_extracted.stat stat"
!yosys -q -p "read_json eval/out/redundant_adders_extracted.json; synth_xilinx -family xcup; tee -o eval/out/redundant_adders_extracted.stat stat"
!yosys -q -p "read_json eval/out/signed_mac_extracted.json; read_verilog eval/blackboxes/dsp_defs.v; synth_xilinx -family xcup; tee -o eval/out/signed_mac_extracted.stat stat"
!yosys -q -p "read_json eval/out/signed_reg_extracted.json; synth_xilinx -family xcup; tee -o eval/out/signed_reg_extracted.stat stat"
!yosys -q -p "read_json eval/out/square_diff_extracted.json; read_verilog eval/blackboxes/dsp_defs.v; synth_xilinx -family xcup; tee -o eval/out/square_diff_extracted.stat stat"
!yosys -q -p "read_json eval/out/unsigned_mac_extracted.json; read_verilog eval/blackboxes/dsp_defs.v; synth_xilinx -family xcup; tee -o eval/out/unsigned_mac_extracted.stat stat"
!yosys -q -p "read_json eval/out/wide_multiplier_extracted.json; read_verilog eval/blackboxes/dsp_defs.v; synth_xilinx -family xcup; tee -o eval/out/wide_multiplier_extracted.stat stat"